In [0]:
# Import Big Sales Data from Volumes

df = spark.read.csv(
    "/Volumes/workspace/default/collegedata/Big Sales.csv",
    header=True,
    inferSchema=True
)

In [0]:
# Select specific columns and filter rows where Item_MRP is greater than 200

from pyspark.sql.functions import *

df_filtered = df \
    .select("Item_Identifier", "Item_Type", "Item_MRP") \
    .filter(col("Item_MRP") > 200) \
    .limit(5)

df_filtered.show()

+---------------+-----------+--------+
|Item_Identifier|  Item_Type|Item_MRP|
+---------------+-----------+--------+
|          FDA15|      Dairy|249.8092|
|          FDU02|      Dairy|230.5352|
|          FDN22|Snack Foods|250.8724|
|          FDP33|Snack Foods|256.6672|
|          FDU02|      Dairy|230.6352|
+---------------+-----------+--------+



In [0]:
# Select specific columns and filter rows where Item_MRP is greater than 200

df_filtered = df \
    .select("Item_Identifier", "Item_Type", "Item_MRP") \
    .filter(col("Item_MRP") > 200) \
    .limit(5)

df_filtered.show()

+---------------+-----------+--------+
|Item_Identifier|  Item_Type|Item_MRP|
+---------------+-----------+--------+
|          FDA15|      Dairy|249.8092|
|          FDU02|      Dairy|230.5352|
|          FDN22|Snack Foods|250.8724|
|          FDP33|Snack Foods|256.6672|
|          FDU02|      Dairy|230.6352|
+---------------+-----------+--------+



In [0]:
# Standardize Item_Fat_Content values to Low Fat or Regular

df = df.withColumn(
    "Item_Fat_Content",
    when(lower(col("Item_Fat_Content")).isin("lf", "low fat"), "Low Fat")
    .when(lower(col("Item_Fat_Content")).isin("regular", "reg"), "Regular")
    .otherwise(col("Item_Fat_Content"))
)
df.show()

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|          FDA15|        9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|Supermarket Type1|         3735.138|
|          DRC01|       5.92|         Regular|    0.019278216|         Soft Drinks| 48.2692|           OUT018|                     2009|     Medium|              Tier 3|Superma

In [0]:
# Fill null values with average for Item_Weight and Unknown for Outlet_Size

avg=df.select(avg("Item_Weight")).collect()[0][0]
df=df.fillna({
    'Item_Weight':avg,'Outlet_Size':'Unknown'
})
df.show()

+---------------+------------------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|Item_Identifier|       Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|
+---------------+------------------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|          FDA15|               9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|Supermarket Type1|         3735.138|
|          DRC01|              5.92|         Regular|    0.019278216|         Soft Drinks| 48.2692|           OUT018|                     2009|     

In [0]:
# Group by Outlet_Type and calculate total sales

df_result = df.groupBy("Outlet_Type") \
    .agg(sum(col("Item_Outlet_Sales")).alias("Total_Sales"))

df_result.show()

+-----------------+--------------------+
|      Outlet_Type|         Total_Sales|
+-----------------+--------------------+
|Supermarket Type2|  1851822.8300000012|
|Supermarket Type1|1.2917342262999993E7|
|    Grocery Store|          368034.266|
|Supermarket Type3|        3453926.0514|
+-----------------+--------------------+



In [0]:
# Remove column Outlet_Establishment_Year from the DataFrame

df_dropped = df.drop("Outlet_Establishment_Year")
df_dropped.show(5)

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-----------+--------------------+-----------------+-----------------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-----------+--------------------+-----------------+-----------------+
|          FDA15|        9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|     Medium|              Tier 1|Supermarket Type1|         3735.138|
|          DRC01|       5.92|         Regular|    0.019278216|         Soft Drinks| 48.2692|           OUT018|     Medium|              Tier 3|Supermarket Type2|         443.4228|
|          FDN15|       17.5|         Low Fat|    0.016760075|                Meat| 141.618|        

In [0]:
# Rename Item_Outlet_Sales to Sales_Amount

df_renamed = df.withColumnRenamed("Item_Outlet_Sales", "Sales_Amount")
df_renamed.show(5)

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+------------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Sales_Amount|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+------------+
|          FDA15|        9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|Supermarket Type1|    3735.138|
|          DRC01|       5.92|         Regular|    0.019278216|         Soft Drinks| 48.2692|           OUT018|                     2009|     Medium|              Tier 3|Supermarket Type2|    443.4

In [0]:
# Convert Item_MRP from double to integer

df_casted = df.withColumn("Item_MRP", col("Item_MRP").cast("integer"))
df_casted.show(5)
df_casted.printSchema()

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|          FDA15|        9.3|         Low Fat|    0.016047301|               Dairy|     249|           OUT049|                     1999|     Medium|              Tier 1|Supermarket Type1|         3735.138|
|          DRC01|       5.92|         Regular|    0.019278216|         Soft Drinks|      48|           OUT018|                     2009|     Medium|              Tier 3|Superma

In [0]:
# Filter rows where Item_MRP is between 100 and 300

df_between = df.filter(col("Item_MRP").between(100, 300))
df_between.show(5)

+---------------+------------------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|Item_Identifier|       Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|
+---------------+------------------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|          FDA15|               9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|Supermarket Type1|         3735.138|
|          FDN15|              17.5|         Low Fat|    0.016760075|                Meat| 141.618|           OUT049|                     1999|     

In [0]:
# Find all items where Item_Type contains the word Dairy

df_dairy = df.filter(col("Item_Type").like("%Dairy%"))
df_dairy.show()

+---------------+------------------+----------------+---------------+---------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|Item_Identifier|       Item_Weight|Item_Fat_Content|Item_Visibility|Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|
+---------------+------------------+----------------+---------------+---------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|          FDA15|               9.3|         Low Fat|    0.016047301|    Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|Supermarket Type1|         3735.138|
|          FDA03|              18.5|         Regular|    0.045463773|    Dairy|144.1102|           OUT046|                     1997|      Small|              Tier 1|Supermarket Type1|         

In [0]:
# Drop rows where Item_Outlet_Sales is null

df_no_nulls = df.dropna(subset=["Item_Outlet_Sales"])
df_no_nulls.show(5)

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|          FDA15|        9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|Supermarket Type1|         3735.138|
|          DRC01|       5.92|         Regular|    0.019278216|         Soft Drinks| 48.2692|           OUT018|                     2009|     Medium|              Tier 3|Superma

In [0]:
# Remove duplicates based on Item_Identifier

df_no_duplicates = df.dropDuplicates(["Item_Identifier"])
df_no_duplicates.show(5)

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|          FDY21|       15.1|         Low Fat|    0.173481304|         Snack Foods| 194.511|           OUT046|                     1997|      Small|              Tier 1|Supermarket Type1|         4910.275|
|          FDW20|      20.75|         Low Fat|    0.040421193|Fruits and Vegeta...| 122.173|           OUT010|                     1998|    Unknown|              Tier 3|    Gro

In [0]:
# Standardize Item_Fat_Content to Low Fat or Regular

df_standardized = df.withColumn(
    "Item_Fat_Content",
    when(lower(col("Item_Fat_Content")).isin("lf", "low fat"), "Low Fat")
    .when(lower(col("Item_Fat_Content")).isin("regular", "reg"), "Regular")
    .otherwise(col("Item_Fat_Content"))
)
df_standardized.show(5)

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|          FDA15|        9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|Supermarket Type1|         3735.138|
|          DRC01|       5.92|         Regular|    0.019278216|         Soft Drinks| 48.2692|           OUT018|                     2009|     Medium|              Tier 3|Superma